# Current-video full diarization and additive overlap extraction

Attach a Kaggle dataset containing the selected YouTube video named `<video-id>_full480.mp4` or `<video-id>.mp4`. An additive Sortformer policy is optional for a new-video baseline run. The notebook preserves the evidence-based baseline; supplemental stages never overwrite it.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile
from urllib.parse import urlparse, parse_qs

VIDEO_URL = 'https://www.youtube.com/watch?v=WopzJl-sulU'
NOTEBOOK_REVISION = 'resumable-window-checkpoints-v32'
REQUIRE_OVERLAP_POLICY = False
RUN_FULL_VIDEO = True
RUN_TARGETED_REVIEW = True
RUN_OVERLAP_EXTRACTION = True
RUN_MOSSFORMER2_REVIEW = True
RUN_CAPTION_GAP_REVIEW = True
RUN_DIAPER_OVERLAP = True
REVIEW_WEAK_CONFIDENCE = 0.35
REVIEW_SHORT_SECONDS = 1.0
BATCH_SIZE = 4

parsed_video_url = urlparse(VIDEO_URL)
VIDEO_ID = (parsed_video_url.path.strip('/') if parsed_video_url.netloc == 'youtu.be'
            else parse_qs(parsed_video_url.query).get('v', [''])[0])
if not VIDEO_ID or any(character not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for character in VIDEO_ID):
    raise ValueError(f'Could not derive a safe YouTube video ID from {VIDEO_URL!r}')

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('Enable a GPU accelerator in Kaggle Settings, then rerun this cell.')
    print(probe.stdout)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN_VALUE = UserSecretsClient().get_secret('HF_TOKEN')
    if not HF_TOKEN_VALUE:
        raise RuntimeError('Add and enable the private Kaggle secret HF_TOKEN, then rerun.')
    print('Hugging Face credentials configured; token not displayed.')
    BASE = Path('/kaggle/working')
else:
    HF_TOKEN_VALUE = None
    BASE = Path.cwd()/'diarization-run'
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'; WORK.mkdir(exist_ok=True)
RESULTS = BASE/('results-'+VIDEO_ID); RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'/VIDEO_ID
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
VIDEO = WORK/('video-'+VIDEO_ID+'-h264.mp4')
REFERENCE = WORK/'target-reference'; REFERENCE.mkdir(exist_ok=True)
print('Video URL:', VIDEO_URL)
print('Video ID:', VIDEO_ID)
print('Notebook revision:', NOTEBOOK_REVISION)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)


## Install and verify

Enable Internet and a GPU before running. The setup uses an isolated environment and verifies CUDA before the full video starts.


In [ ]:
import base64
import zlib
SOURCE_ARCHIVE = '...'
EMBEDDED_FILES = json.loads(zlib.decompress(base64.b64decode(SOURCE_ARCHIVE)))
for name, source in EMBEDDED_FILES.items():
    (WORK/name).write_text(source)

ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib-cache')
ENV['MPLBACKEND'] = 'Agg'
ENV['NUMBA_CACHE_DIR'] = str(BASE/'numba-cache')
if HF_TOKEN_VALUE:
    ENV['HF_TOKEN'] = HF_TOKEN_VALUE
    ENV['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN_VALUE
def checked(command, **kwargs):
    return subprocess.run(command, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking the isolated environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy(); bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)

print('2/4: Installing the pipeline and extraction model', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt',
    'yt-dlp', 'soundfile', 'safe-gpu', 'yamlargparse==1.31.1',
    'decorator', 'h5py', 'matplotlib', 'librosa', 'scikit-learn', 'tensorboard',
]
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', 'clearvoice==0.1.2'])
checked([PYTHON, '-m', 'pip', 'install', 'gdown', 'librosa==0.10.2.post1',
         'rotary-embedding-torch==0.8.3', 'scenedetect==0.6.6',
         'python-speech-features==0.6', 'torchinfo', 'pydub'])
checked([PYTHON, '-m', 'pip', 'install',
         'git+https://github.com/wenet-e2e/wespeaker.git'])
# WeSep's current package metadata omits its namespace-style wesep/utils
# directory. Keep the checkout and put it first on PYTHONPATH so the complete
# source tree is used, while pip still installs all declared dependencies.
WESEP_SOURCE = WORK/'vendor'/'wesep'
WESEP_REVISION = '99eca54b60300d39b9353d93cf285a14bba37854'
wesep_revision_file = WESEP_SOURCE/'.codex-compatible-revision'
if (not (WESEP_SOURCE/'wesep'/'utils'/'utils.py').is_file()
        or not wesep_revision_file.is_file()
        or wesep_revision_file.read_text().strip() != WESEP_REVISION):
    if WESEP_SOURCE.exists():
        shutil.rmtree(WESEP_SOURCE)
    WESEP_SOURCE.parent.mkdir(parents=True, exist_ok=True)
    checked(['git', 'clone', '--no-checkout',
             'https://github.com/wenet-e2e/wesep.git', str(WESEP_SOURCE)])
    checked(['git', '-C', str(WESEP_SOURCE), 'checkout', WESEP_REVISION])
    wesep_revision_file.write_text(WESEP_REVISION + '\n')
checked([PYTHON, '-m', 'pip', 'install', str(WESEP_SOURCE)])
# The upstream wheel omits wesep/utils because that directory has no
# __init__.py. Overlay the complete checkout onto site-packages so imports do
# not depend on PYTHONPATH or notebook process state.
site_packages = Path(subprocess.check_output(
    [PYTHON, '-c', 'import site; print(site.getsitepackages()[0])'],
    env=ENV, text=True).strip())
installed_wesep = site_packages/'wesep'
shutil.copytree(WESEP_SOURCE/'wesep', installed_wesep, dirs_exist_ok=True)
missing_utility = installed_wesep/'utils'/'utils.py'
if not missing_utility.is_file():
    raise RuntimeError(f'WeSep repair failed; missing {missing_utility}')
# These upstream source directories contain Python modules but omit package
# markers, which is also why they disappear from the built wheel.
for directory in [installed_wesep/'utils', installed_wesep/'dataset', installed_wesep/'bin']:
    if directory.is_dir():
        (directory/'__init__.py').touch()
ENV['PYTHONPATH'] = str(WORK) + os.pathsep + ENV.get('PYTHONPATH', '')

print('3/4: Selecting the CUDA ONNX runtime', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg is required. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')

print('4/4: Verifying GPU imports and focused behavior tests', flush=True)
verification = """import torch,onnxruntime as ort,wrapt,wesep
import chainofrules,repeat_evidence
from wesep.models import get_model
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX providers:', ort.get_available_providers())
print('WeSep repaired package:', wesep.__file__)
assert get_model('BSRNN').__name__ == 'BSRNN', 'WeSep English checkpoint is incompatible'
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is unavailable'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX runtime is unavailable'\n"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers',
         'test_repeat_evidence', 'test_overlap_resolution',
         'test_overlap_extraction_review', 'test_confident_transcript',
         'test_mossformer2_review_policy',
         'test_reference_promotion', 'test_diaper_overlap'], cwd=WORK)

import hashlib
REFERENCE_FILES = {'face_embeddings.npy': '...',
 'opening_officer_reference.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE5MiwpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIApDR569eD+ePSVTQjzXL8Q8mFepvQvxDz3M/vy9YNoYvbW3or2vrIm9B/1hvZUsuD3xL569DlK0vZXXV76CoP49pTbJvXm8+D0rOMO7dQlDPgHbrb32ZTq9CKD5vQ9U37tiJZm9awdSPA8JgL2s9lw9o0yAPS0WeT0kG5e9Qyscvnywsr0gUcM9MYe9PWgwhT2T9/+8dN+VPE0xhL0qjMe9kvxuPYCZDL4TrUS9GluQPZgFOD1fWoc9hl6NPSuuzj07pmY9G28RvUyrVrx3J4o9nemUvf1lCb2RV1O9samYOx9Kwz3HZcS83eeqPRlqHrx3YXi9tTCWvIBTfT3P+nq97hzyu3W4qT1uji49wXiDPatGqT3xJ6Y9+7ytvRKQeD0oG5W9AEVSO0Rk2z0s18S8yngZvqcF+TwuJk07nuqgvFu6i7zfPdk8XlA7vIzIU72kDnQ7LuPhPflzkTy7ZaU9SQprvOXkfz2Rcaa971UaveED+r1wB3E9oWuXvfqJ6b04ZL+8glvUvf9vVzvUXMy9PJmRvGJhQz04Ou49Me+yvV+DsL3XHok97akwOyr1q7x9JoE8t7utvXZgrT1+4sS97rJQvWCahrzGHao9BmubPWbicbydw8o9UoElPaMZxT2Kjim8lFDIPAeF873jfsk9KHSFPQCjizwX4sw8R/MEPahYvr1tTkC9i+NzPR4WEz0djJY8bPG+PL661j2XY0S8ZmstPcr++r0yOwc8hiOaPAC8ar3XBI68ty0JvAQC2LyIdd69/njeu/eipT3X4pE9lrucvL/j3D0f7uA9prS+PRFYW71r4v+7YbtYPRYMaD09C4u8QvxFPX7KIb1GAbE99asPvp3whr3hlR46/mkAPvSJrTyyiaW8tCXrPKVQu7xzO7+96JEzPevqjjydtB6+tr9tPdb7Kr2T0hs9InaLPXhmhb1qw0Y9pWg+vdvlEr2TkHA9JC7gvdhyML0Dtlc95fy6PVnmz71jg+O9MjXBPethZz0e+hg9OvCCPXPA3ro=',
 'reference.json': 'ewogICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgIm1ldGhvZCI6ICJyZXZpZXdlZCBwb3N0LXJ1biBwcm9tb3Rpb247IHZlcnNpb25lZCBhbmQgcmV2ZXJzaWJsZSIsCiAgInBhcmVudF9yZWZlcmVuY2UiOiB7CiAgICAicGF0aCI6ICJyZWZlcmVuY2VzL2F1ZGl0b3Ita2FnZ2xlLXYxMy1wYXJlbnQiLAogICAgInZvaWNlX2VtYmVkZGluZ3Nfc2hhMjU2IjogImRiNTcwMTE4ODAwNjBhN2Q4NmY0YjU5ZmU4MjVkYzM5ZTFjZTE3YjZmN2NkNWJmMmI1ZDczNGI4ODFhYmE2ZTAiLAogICAgInJlZmVyZW5jZV9qc29uX3NoYTI1NiI6ICI1YTI2NjNlMjZhNjEyNWMwMTEwMDYzZTk3YzY4NGFkY2QyMjNkMzA3ODVmNDUzZDRhYTQ0MGJjMGM2NjhmZWFhIiwKICAgICJtZXRhZGF0YSI6IHsKICAgICAgIm1ldGhvZCI6ICJtYW51YWwgaWRlbnRpdHkgYXBwcm92YWwgZm9sbG93ZWQgYnkgY29uc2lzdGVuY3kgc2NyZWVuaW5nIiwKICAgICAgInNjcmVlbmluZyI6IHsKICAgICAgICAidm9pY2UiOiB7CiAgICAgICAgICAiYW5jaG9yX3JvdyI6IDQsCiAgICAgICAgICAiYWNjZXB0ZWRfcm93cyI6IFsKICAgICAgICAgICAgMCwKICAgICAgICAgICAgMSwKICAgICAgICAgICAgMiwKICAgICAgICAgICAgMywKICAgICAgICAgICAgNCwKICAgICAgICAgICAgNSwKICAgICAgICAgICAgNiwKICAgICAgICAgICAgNywKICAgICAgICAgICAgOSwKICAgICAgICAgICAgMTAsCiAgICAgICAgICAgIDExLAogICAgICAgICAgICAxMiwKICAgICAgICAgICAgMTMsCiAgICAgICAgICAgIDE0LAogICAgICAgICAgICAxNQogICAgICAgICAgXSwKICAgICAgICAgICJleGNsdWRlZF9yb3dzIjogWwogICAgICAgICAgICA4LAogICAgICAgICAgICAxNgogICAgICAgICAgXSwKICAgICAgICAgICJzaW1pbGFyaXR5X3RvX2FuY2hvciI6IHsKICAgICAgICAgICAgIjAiOiAwLjY1MzI3Mjc0Nzk5MzQ2OTIsCiAgICAgICAgICAgICIxIjogMC42NjgwODI0MTYwNTc1ODY3LAogICAgICAgICAgICAiMiI6IDAuNjMwNjA5MDM1NDkxOTQzNCwKICAgICAgICAgICAgIjMiOiAwLjY0NjczNDM1Njg4MDE4OCwKICAgICAgICAgICAgIjQiOiAxLjAwMDAwMDIzODQxODU3OSwKICAgICAgICAgICAgIjUiOiAwLjY5NjUyNTgxMjE0OTA0NzksCiAgICAgICAgICAgICI2IjogMC42NjY2NzAzMjI0MTgyMTI5LAogICAgICAgICAgICAiNyI6IDAuNjk0NDA0MzYzNjMyMjAyMSwKICAgICAgICAgICAgIjgiOiAwLjM4NzU2NTA3NjM1MTE2NTc3LAogICAgICAgICAgICAiOSI6IDAuNTM2NDkyNDA3MzIxOTI5OSwKICAgICAgICAgICAgIjEwIjogMC41NTY0OTY4NTg1OTY4MDE4LAogICAgICAgICAgICAiMTEiOiAwLjU2NTQ1MDk2NjM1ODE4NDgsCiAgICAgICAgICAgICIxMiI6IDAuNTEyNjA2NjgwMzkzMjE5LAogICAgICAgICAgICAiMTMiOiAwLjU5NTc1NjExMzUyOTIwNTMsCiAgICAgICAgICAgICIxNCI6IDAuNDc3MTU5MzgwOTEyNzgwNzYsCiAgICAgICAgICAgICIxNSI6IDAuNTY5MTMyMzI4MDMzNDQ3MywKICAgICAgICAgICAgIjE2IjogMC40MjcxOTU3Mjc4MjUxNjQ4CiAgICAgICAgICB9LAogICAgICAgICAgIm1pbmltdW1fc2ltaWxhcml0eSI6IDAuNDUKICAgICAgICB9LAogICAgICAgICJmYWNlIjogewogICAgICAgICAgImFuY2hvcl9yb3ciOiAxOSwKICAgICAgICAgICJhY2NlcHRlZF9yb3dzIjogWwogICAgICAgICAgICAyLAogICAgICAgICAgICAzLAogICAgICAgICAgICA0LAogICAgICAgICAgICA5LAogICAgICAgICAgICAxMCwKICAgICAgICAgICAgMTEsCiAgICAgICAgICAgIDEyLAogICAgICAgICAgICAxMywKICAgICAgICAgICAgMTQsCiAgICAgICAgICAgIDE1LAogICAgICAgICAgICAxNiwKICAgICAgICAgICAgMTcsCiAgICAgICAgICAgIDE4LAogICAgICAgICAgICAxOSwKICAgICAgICAgICAgMjAKICAgICAgICAgIF0sCiAgICAgICAgICAiZXhjbHVkZWRfcm93cyI6IFsKICAgICAgICAgICAgMCwKICAgICAgICAgICAgMSwKICAgICAgICAgICAgNSwKICAgICAgICAgICAgNiwKICAgICAgICAgICAgNywKICAgICAgICAgICAgOAogICAgICAgICAgXSwKICAgICAgICAgICJzaW1pbGFyaXR5X3RvX2FuY2hvciI6IHsKICAgICAgICAgICAgIjAiOiAwLjQxNzU0ODA2MDQxNzE3NTMsCiAgICAgICAgICAgICIxIjogMC4zNTYxNzQ3MDc0MTI3MTk3LAogICAgICAgICAgICAiMiI6IDAuNjY0NjU4ODQ0NDcwOTc3OCwKICAgICAgICAgICAgIjMiOiAwLjYxMTAzNjM2MDI2MzgyNDUsCiAgICAgICAgICAgICI0IjogMC42MDYwNTc4ODIzMDg5NiwKICAgICAgICAgICAgIjUiOiAwLjQxNzEwNDYwMTg2MDA0NjQsCiAgICAgICAgICAgICI2IjogMC40MTk0MjAyNzIxMTE4OTI3LAogICAgICAgICAgICAiNyI6IDAuMzk2NDAyNjU3MDMyMDEyOTQsCiAgICAgICAgICAgICI4IjogMC40MzU0NDgxMTAxMDM2MDcyLAogICAgICAgICAgICAiOSI6IDAuNDkzMzMwMzU5NDU4OTIzMzQsCiAgICAgICAgICAgICIxMCI6IDAuNTE3MTQ2OTQ0OTk5Njk0OCwKICAgICAgICAgICAgIjExIjogMC40NjExMjI2OTE2MzEzMTcxNCwKICAgICAgICAgICAgIjEyIjogMC41MDkzODgxNDg3ODQ2Mzc1LAogICAgICAgICAgICAiMTMiOiAwLjczNjIwOTYzMDk2NjE4NjUsCiAgICAgICAgICAgICIxNCI6IDAuNjc1NTUyOTA0NjA1ODY1NSwKICAgICAgICAgICAgIjE1IjogMC42MzA3NDkzNDQ4MjU3NDQ2LAogICAgICAgICAgICAiMTYiOiAwLjY5NTE5MDE5MTI2ODkyMDksCiAgICAgICAgICAgICIxNyI6IDAuNTk5NTAwMjk4NTAwMDYxLAogICAgICAgICAgICAiMTgiOiAwLjgzMzAwNzkzMTcwOTI4OTYsCiAgICAgICAgICAgICIxOSI6IDEuMCwKICAgICAgICAgICAgIjIwIjogMC45Mjk5MDYzMDg2NTA5NzA1CiAgICAgICAgICB9LAogICAgICAgICAgIm1pbmltdW1fc2ltaWxhcml0eSI6IDAuNDUKICAgICAgICB9CiAgICAgIH0sCiAgICAgICJ2b2ljZV9zb3VyY2VzIjogWwogICAgICAgICJkOTVkYjU1NzkxZWE0YmI2OWQ5YTRjYjYxNzg2NWQ0ZCIsCiAgICAgICAgIjVjYzgwNjQ5YzQwZjRmMWFhYTU4N2U3NTQ0MmU3ZjQzIiwKICAgICAgICAiOTgwYmI5MTZlZjhiNDlmMGFlYTdkZTU4ZmY4NWM4MWQiLAogICAgICAgICIwMjc0NjE2M2IzZDI0N2UzOTBjZGRkMWMxZmFmNjIzZCIsCiAgICAgICAgImQyZDE2ZDVmNzE5MjQ4Y2ZhOGRhZDExODk3Mjc5MWRmIiwKICAgICAgICAiNTZkOTA0MDA4Mjk4NGFkZTk4Mzg3YmU0NjhiMDk1OTEiLAogICAgICAgICJiMWIyMDZiZTk4ZDc0ZmJkOTZhZWNjY2VhZWMwYWMwOCIsCiAgICAgICAgIjQ0ZTZlODA1YmVlYzQyOTQ4MmY1YzVkNWQzODliZTkzIiwKICAgICAgICAiMTFhZWNhNzg0ZDgyNDRmZThkODkzNTRkMTM2MzM1ZDgiLAogICAgICAgICJhZWI1YmU4OTFkOTU0YjY1ODZmYWY0ZWYyNzQxNTViZiIsCiAgICAgICAgImZmOTM0Nzc3MzViNDRmYzFiNzBjMDc2MTYzZjljNTkyIiwKICAgICAgICAiNjhhNGI2ZTgwM2I1NGQxM2FjNGI2YTFiMGJhOWY4OTIiLAogICAgICAgICI5YWZlZGFlZDUzMTE0NWIwOTVkMThmYjE4NmU2ZmU5OCIsCiAgICAgICAgIjcyNDJhNWUxMzFlNTQ0MjNhZDg0YWIzMTQxZmY5ZWQyIiwKICAgICAgICAiZTdiNzJlZTkxNzRkNGZmOWE5OTgxYzJhYWNkZjE0OWQiLAogICAgICAgICIzMjE0ODVjNWI2Y2E0ZWY0ODU1MjFmYTMwMjNlOTgwYyIsCiAgICAgICAgIjNiNzhjMjNmNzFiZDQ0NGY5NzcxNWUzODAwMWNkNDg1IgogICAgICBdLAogICAgICAiZmFjZV9zb3VyY2VzIjogWwogICAgICAgICJkOTgyNmZlNzZmNDM0MmMxODdiNjljZDI0YjM0NDQyYyIsCiAgICAgICAgImU2MjhjYmEzN2VjNjQzMjRhMWRiNjg0MmQzMzFkNDBmIiwKICAgICAgICAiNTY3MmQ4NWM2Njg2NGE2MzhhNDU1NGRlNWIwZWU0OGIiLAogICAgICAgICI4MzUxYTdjYzQ1Y2Y0MzM5OWZiYTk5YWYyMmFlNzliZSIsCiAgICAgICAgIjE5ZGQxZGZmY2U2NDQ0OGZhYjcyOWIwYWFlNWJmNDJhIiwKICAgICAgICAiYTlmNzI2NTc4ZDliNDFkY2JmNTQzMjFjYjU4NmY1NDUiLAogICAgICAgICJmYzA5YmFkZDQwZjA0NGJiYjQ2ZDgxZTVmNmNjZTVmMCIsCiAgICAgICAgIjEzOGMwMzhiM2I5MTQ0MjZiOWJlZWE1MGU3OTRkZDhiIiwKICAgICAgICAiOTI3NzI2ZjNiYWUzNDU3OTk4ZDFmNTE1NzZmNDNkYmYiLAogICAgICAgICI2NWQ1MzMyYTkwNDU0YzZkOWY1MzM4Y2I5OWU1NTljZiIsCiAgICAgICAgIjUwY2FiZDk3NzQ3YjRlMWI5Zjc2YWYxZWE5YTg4YjVhIiwKICAgICAgICAiNzAzNWU0MmUwMmE5NGI1Zjk3YmZjODNjMDRlYzhmOWQiLAogICAgICAgICJhZWI1YmU4OTFkOTU0YjY1ODZmYWY0ZWYyNzQxNTViZiIsCiAgICAgICAgImZmOTM0Nzc3MzViNDRmYzFiNzBjMDc2MTYzZjljNTkyIiwKICAgICAgICAiNjhhNGI2ZTgwM2I1NGQxM2FjNGI2YTFiMGJhOWY4OTIiLAogICAgICAgICI5YWZlZGFlZDUzMTE0NWIwOTVkMThmYjE4NmU2ZmU5OCIsCiAgICAgICAgIjcyNDJhNWUxMzFlNTQ0MjNhZDg0YWIzMTQxZmY5ZWQyIiwKICAgICAgICAiZTdiNzJlZTkxNzRkNGZmOWE5OTgxYzJhYWNkZjE0OWQiLAogICAgICAgICIzMjE0ODVjNWI2Y2E0ZWY0ODU1MjFmYTMwMjNlOTgwYyIsCiAgICAgICAgIjkwMDk3MjlmOTNmODQ2ZjNiMDA2M2FjZmYyZDMwN2Q5IiwKICAgICAgICAiM2I3OGMyM2Y3MWJkNDQ0Zjk3NzE1ZTM4MDAxY2Q0ODUiCiAgICAgIF0sCiAgICAgICJzaG9ydF9saWJyYXJ5X2lkcyI6IFtdLAogICAgICAibm90ZSI6ICJObyBpbmZlcmVuY2UgaXMgYW4gYXBwcm92YWwuIFNob3J0IHNhbXBsZXMgZXhjbHVkZWQgZnJvbSB0aGUgbWFpbiB2b2ljZSBjZW50cm9pZC4gSG9sZCBldmFsdWF0aW9uIHZpZGVvcyBvdXQgb2YgZW5yb2xsbWVudC4iCiAgICB9CiAgfSwKICAicHJvbW90aW9uX3JldmlldyI6IHsKICAgICJtYW5pZmVzdF9zaGEyNTYiOiAiYjQwMzkxYjE4ZWFhYWJkNjgzMWYyZjUyNzBkNmMzMWU4YWZkMGIxMTVjMmQ3OThjNmI0NjkxNTI4MGYxMTAxZiIsCiAgICAiYXBwcm92YWxzX3NoYTI1NiI6ICIxMTQ4NmU4MzBlMjNjNjEwOTJlMDJkYzUxNjM2NWFlNWU1NTJiNDVhMDJiMTc5NWI3MmRkZDdiZDk1MDNkN2RhIiwKICAgICJwcm9tb3RlZF9jYW5kaWRhdGVzIjogWwogICAgICB7CiAgICAgICAgImJhc2VsaW5lX2luZGV4IjogMTcsCiAgICAgICAgInN0YXJ0IjogNDguNjA4LAogICAgICAgICJlbmQiOiA1MC4xNywKICAgICAgICAiZHVyYXRpb24iOiAxLjU2MjAwMDAwMDAwMDAwNDcsCiAgICAgICAgInRleHQiOiAiSSdtIHN0YW5kaW5nIGhlcmUgc2F5aW5nIEdvZCBibGVzcyBob21lbGVzcyB2ZXRlcmFucy4iLAogICAgICAgICJyYXdfc3BlYWtlcl90cmFjayI6ICJTUEVBS0VSXzA0IiwKICAgICAgICAiZmluYWxfY29uZmlkZW5jZSI6IDEuMCwKICAgICAgICAicmVmZXJlbmNlX3NpbWlsYXJpdHkiOiAwLjU4Mzk5NDI2OTM3MTAzMjcsCiAgICAgICAgImxvY2FsX3ZvaWNlX3N0cmVuZ3RoIjogMS4wLAogICAgICAgICJyZXZpZXdfcmVxdWlyZWQiOiB0cnVlLAogICAgICAgICJhdWRpbyI6ICJhdWRpby9jYW5kaWRhdGUtMDAxNy00OC42MDgtNTAuMTcwLndhdiIsCiAgICAgICAgImF1ZGlvX3NoYTI1NiI6ICJlZmJhN2E3MjkzMjhhMWIzN2ZmZmE2Yjc1ZmQyOGJlMzEyZmUxNjlhOThkZGM1NTVmNzE4ZjNmNTM1MThkNzIxIiwKICAgICAgICAic291cmNlIjogewogICAgICAgICAgInZpZGVvIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9dUF0aUV2aVV6R0EiLAogICAgICAgICAgInN0YXJ0IjogNDguNjA4LAogICAgICAgICAgImVuZCI6IDUwLjE3CiAgICAgICAgfQogICAgICB9LAogICAgICB7CiAgICAgICAgImJhc2VsaW5lX2luZGV4IjogMTkyLAogICAgICAgICJzdGFydCI6IDU0NS40MjksCiAgICAgICAgImVuZCI6IDU0Ny40MzEsCiAgICAgICAgImR1cmF0aW9uIjogMi4wMDIwMDAwMDAwMDAwNjY0LAogICAgICAgICJ0ZXh0IjogIllvdXIgcXVhbGlmaWVkIGltbXVuaXR5IGlzIG5vdCBnb2luZyB0byBzdXJ2aXZlIHRoaXMuIiwKICAgICAgICAicmF3X3NwZWFrZXJfdHJhY2siOiAiU1BFQUtFUl8wNCIsCiAgICAgICAgImZpbmFsX2NvbmZpZGVuY2UiOiAxLjAsCiAgICAgICAgInJlZmVyZW5jZV9zaW1pbGFyaXR5IjogMC41MzM2OTgzNzk5OTM0Mzg3LAogICAgICAgICJsb2NhbF92b2ljZV9zdHJlbmd0aCI6IDEuMCwKICAgICAgICAicmV2aWV3X3JlcXVpcmVkIjogdHJ1ZSwKICAgICAgICAiYXVkaW8iOiAiYXVkaW8vY2FuZGlkYXRlLTAxOTItNTQ1LjQyOS01NDcuNDMxLndhdiIsCiAgICAgICAgImF1ZGlvX3NoYTI1NiI6ICIxMWZjMDA0NjM0MWRjNzFiNzkwOGQ1ZTM4NTY3MWU5YjQ5YzAwNzExM2QxNGIxN2JmM2I1ODk2OTQzMjFjNDI0IiwKICAgICAgICAic291cmNlIjogewogICAgICAgICAgInZpZGVvIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9dUF0aUV2aVV6R0EiLAogICAgICAgICAgInN0YXJ0IjogNTQ1LjQyOSwKICAgICAgICAgICJlbmQiOiA1NDcuNDMxCiAgICAgICAgfQogICAgICB9CiAgICBdCiAgfSwKICAidmFsaWRhdGlvbiI6IHsKICAgICJwYXNzZWQiOiB0cnVlLAogICAgImNoZWNrcyI6IHsKICAgICAgImFsbF9jYW5kaWRhdGVzX21hdGNoX3BhcmVudCI6IHRydWUsCiAgICAgICJjZW50cm9pZF9zaGlmdF9pc19ib3VuZGVkIjogdHJ1ZSwKICAgICAgImV4aXN0aW5nX3JlZmVyZW5jZV9hZmZpbml0eV9pc19wcmVzZXJ2ZWQiOiB0cnVlCiAgICB9LAogICAgInRocmVzaG9sZHMiOiB7CiAgICAgICJtaW5pbXVtX3BhcmVudF9zaW1pbGFyaXR5IjogMC41LAogICAgICAibWluaW11bV9jZW50cm9pZF9zaW1pbGFyaXR5IjogMC45OTUsCiAgICAgICJtYXhpbXVtX2V4aXN0aW5nX21lZGlhbl9kcm9wIjogMC4wMQogICAgfSwKICAgICJjYW5kaWRhdGVfc2ltaWxhcml0eV90b19wYXJlbnRfY2VudHJvaWQiOiBbCiAgICAgIDAuNTgxMTYxMDIyMTg2Mjc5MywKICAgICAgMC41MzE4Mjk5NTMxOTM2NjQ2CiAgICBdLAogICAgIm9sZF90b19uZXdfY2VudHJvaWRfc2ltaWxhcml0eSI6IDAuOTk1NzA0OTQ4OTAyMTMwMSwKICAgICJleGlzdGluZ19yZWZlcmVuY2VfbWVkaWFuX2FmZmluaXR5X2Ryb3AiOiAtMC4wMDIxMzc3MjA1ODQ4NjkzODQ4CiAgfSwKICAidm9pY2UiOiB7CiAgICAicGFyZW50X3Jvd3MiOiAxNSwKICAgICJwcm9tb3RlZF9yb3dzIjogMiwKICAgICJ0b3RhbF9yb3dzIjogMTcsCiAgICAiZW5yb2xsbWVudF9zaGEyNTYiOiAiNDY2ZTg2MmRjMTU5MTVhMzFlZjk5MzJjZWZjM2VlNjZlYTY1MGY2Y2E2YmEwOGEzZDkwYmU3OGQxMjFjMTlkNyIKICB9Cn0K',
 'voice_embeddings.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE3LCAxOTIpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAryYYA9E3/UPKH0CD7MhyU9xdnRPW10mT0daXQ5ahlPPYpuob2c04O9luHlPP9rbz0a46y9mgDPPZknEz4Beni9hyhlPfRCj7vrA4y9itULvslpAT3tH489TvS/u+Hd5jzRNBO82H4ivTIZjT2+seC8MHu7PGww4L0p1yK9fpBDPaf23byFqDA8YlwqPkJlo709Ssq9GN2uu4+ynb0Y7jE9+04PvuVCxD1XbCI9+0VxPbu5CryWeby9nGhEveePpL2thSK892Squz16Zb342KC9a41Vupy3e7ysHRg+7RnDvC90or0/kBa9AkjIPZ70Br6Tx2+94YKzvcES3L3IAsY8gNXIPT0GJL3rrMc4YlCWvT7HKz2LBZ48uD6+vCi50z0UKky8ZnkpvQQj/T3sOo29Mc1lvXhh+r1uyXM8Q/NzvcWvPj0m+cO8TrLZPdUlwT0dK8U9OVK1vYpkRr4/pVQ9icuiPWuRXj3Eq0O8Q35CO2dyKj3SDng9z419vZC4JD2dfJy9bMx2vbXUJj5VW/k9BMs3vRv7IL1nJw497TaXvNuos7zBfGC+nI/HvfX0cb0F+RU9IDYrPk3foD3wpra8Rr3fvcpDgT0Sd548KO1TPexPTT3EpT887TLZPD2PQLyW8MW8JR6LvdEztj02oqY8LSyCvUG8Vb7mere8HoOxvAsriTxRA1a8M+2cvJHnnj3JRL+93+1BPYRVmjwAmQK8OZCHvZu2pT3b9La8MUacOo6hkz3e8wY9jPmsvZuOyL1phi49VDESPUjaFj02F0A9uIjevHLjA70tptE8pgjQvOI8lT2PQW49FMBYPQgBWj0gOee91YDXPQOUgL0zNZs9A3G5PeWCaDxe/uk8nnhLvfYIZr0IGD293YSyPYX5Aj5DguI8tuQZPBEfhD28CnU9NtdlvLncEr6aXsu8a5miPe+xrT0p+LQ9AjT/vCKTKz3ZUFs9U4cXPMgDuz0aF+g8WmQMvZJFgb1xtAW9Vu2+PNG21j3czgG93t8WPQmI+TzOs8c8mPqpuy9L0T3Yrz09Hs4EvZbblT3UEmS9OfLoPWwsLr2Wcti91u5OPRT0Nz1f9De9AMwjPTLBHD6k9RG944kIPkGjhz0iYpa82gPaveyVgDvmCgM93AaAvFgeTb3J4sK8jDYovTDMVj27gvQ8YJ66PfuypL2KBFi89PDHvLUfzTyD8qw8daZFPoO8IL7HlzO9DMeXvFnEz72p7Qm9kbbyvY7jyD3hQg09gglqPSGZqTxYoGy9RJc5PBinkr0TjLM8KD4rvZ1LBb5RW6i9uLayPGWTDr2mqvA8nJ+ivYYZKL3ZDUu95KauPaH4p72mSkg98LahvaL4x71mPpA9euNFPaAgnDyHiza9S4/OvNa2O72IT4s8K7H8vE4miz2HMQM95iXsvd0hoD1pzqS9OLl3vbEc3L1dsoc9pFvMvD950T3m1HG9V8P8PTSYlj2+cMU9qbWovUqfCb5iEqo9m3mRPULHsb1y8P+8g7IlPSlOxTxuIrw8sIeZvbf1KL2r4vS8ziSUvTlsdT4JuxQ+PBIRPQ0xs72nG/M9nMwDvGoUE71utwm+XtASvi+rtb1mzYs9QhEDPh+qCT7kpSW9QPE8vbWewjzudd098keHu/swBL2XTpo9uNqlvHorZL2WgQW9YnkNvvAHiD3VYpY8Nh+lvYbdAr5Tc708ibMFvVfi5T04zhI9czzEvTl+DT009Nu9PBSTPe12YL2mKH69b2oAvbBVcT09ylO9A9AwvMhHSb1/RI89HROvvVdiVb3rd5M8V5d0PeauWL2nge27d6+MOzg2OT0lNDU93+Q0PBRL9z0WCtO85gEoPUw5SbwrNK+9GcOmPbMmw711G589NRtlPWSrfrybA7k8SKXJvGWUBr0YJIG9nG+ZPXzDtD36fpk9xrZjPD0Bwz1U+YS8MvUEvkSmgr1WArS9dBiOPel8CT0nMC08zymovPzwsz3DTo89VCi7uiVzJDx04k09vG8qPLkBe70HpUe9Fq06PRZvoD1djpi9EvBaPLLuJD1TJnQ9dfImvDx2Rj0bGZc8XDsmvIXusz186729+MUVPiGDAr202bC9TeQCPt5sUDyhhdW9HqNjPSZXmD3H9Hq9c/DIPZBzFL3/fH68L522vRwRpzxIPUE9RXvTPPQ9Bzyx7+Q8UC5EvbfWGz6qWT894laiPRJOHr5vWgu7kdp/vPZArbzaq583csKCPgVz1b2SSoi9OUGBvQyZpL2Lo9K92xoovlVyWzz5B6e8ffvyuVszID15xju+Sh51vfuuuL1I5Vs9fL95vf+6ir3EyEW9ZhQdPKTktrwi2xQ+vdG0vcjGM71BbgS9w+rlOvsyXb2O8UQ8QxWyvAcltjy6gOg8m2lWPbr8e72A3DY9w7cMvQ8Ibr0XipE85L4avHBQTz2LeYs94LWrvbQbrT01f+W9PTkwPSBjmr1Ljym9fdrovDjsXT3agNe83W9WPc28sz0U4gY+QQ9BvWuNOr2N2AM9UaZUPQb7mr02SC09SCQYPahTaj3by/Y9g2kPvnPl/7wad5M9xzQ3vdd+Bj4f6XQ9rs6NvbePPzyKrOs8cU32vW1/grw1xSe+HScGviBeML3Fbtu8YUupPUMrAT7Q84496mYTvhs2/zyKjfc8DxH8vMxsez26PBc8Jb8ePd8zv7wk86U7ArRPvNiNj7z9xxO9QJTYvZ0D8r1XS1e9Nr9EPKPKAb1iTYg8c005vYL4TD3AMcO8Ogi5PK1XijyDrHy90J+EvQIxPj03Xe69jHbFu9UfCz0+LnI91eabvbh+ub0gG4A9CqCOPbU0mz1FNSm902ndPIjypr1ymwA928lXPa/Q7j2BB+o83EibPZ+eBj7auiO+Ud2CPfnDjL0pzrg9pZvKujGVwj3sPH26BqP3u4pyFr2eKqK9elJROx1YkT0j5z49EklLvbvVGT6haU49Wn2wvfDEfb0BPGU7gsJuPH12Oj3k+Gc9feLYPP6dHT5KZz89+iK1PDLwkD0W0iq9oaD8PDHVuroO8ae9nWfGPJJ9cz23edK8pyLDvHUv7jxIQA49BwEmOlXBkj2gVo09chmmPBhf9z3UVZO968VTPYBJGr6fE9y9CfLzPX4eDj4vkpy9DF3xPVZvgj09aLa9OmnaPfb4L70MggY7Ge4uvvVlET3CM4c9C+A1vL5RVL1zXcM8cy3oPEz3iz06VIk80w41PfyZJr3xgJI9xE7DvJ2jCz347Lk8RPodPt0sjb2m/pS9Q+govH4tub0F9IG9OfEOvixRMTwjBRy97k49PZB00DwFWuG9L9bcvCDrh73ii+E9TjxevdsvS7ySfzG9wv5vO9rGpL2PRrQ9ChKvvQJQprxz1qS8WS3BPUYAsL09Uma7E03Ovd1u3713Ux89vk6NPNKWmbzbC2k8Ly4yvcgGpD1uIje9XgWJu84noTt67H48J7AAvrMRnz3U9eO8jLCjvEvcQzuvckA9ci4lPJXUeD3/jI67P6LjPZTuTj3UYTA+035pve4TIL6Zo/c9DbSbPactl70/EKw7yFm8PQEAhr0gFn08Ts0vPPjrPr1sLJU84BNXvbQwTD5GoEQ+HhcOvAxsbbyNzxY8ykiNvbjXi7xWU6K9YphnvH8AXL3QGTq6JxktPpzkhT0ve2o878R9vbqTTD0Nq948drCrvWPaWTy+iaC8MhWRPelfvzvoPx29yqmCPO8WULzpmnw8+eswvYVORr6nBac9Si98vA38Oj0VqBU8xYQSvisddj04r/29XuCGPZrm7Tyzemm96BQtvdA8oT0IlMc8kfpAvG4Hqj3xSNU9tq6lvRk/a71RG7c9Ue4GuzJZZjyliSO9OO3yvIFFIj1AQ3494/gAPVww7zzxX4s6vBBzPdvBxD0/kzC+v+ffPazcCL6Po1A9u4aHPS/9JDxLw1W9s7wIvC0Pl7yc07G9NYmkPSNcxj200Xs9DbIqPBV9gz2x5K095oDQvepU+73SI1q9yQibPEf/wjwFXQk9mhpLO8l68j3gasA8+PG7PIhQWLxJb9k8wAFKvTGEjL18Ghq96lYYPQXl2Dw+I1E7qeaYOgKMH7xnscm80llsPWWrhz2Doj4962znPI2A2jzxjyY8azQwPW4yQjzhs1q89OnCPDIsID6twyS8m7AHPlhibT0B/4+9COWbPYhlBrojv1S9WTrVvdFmaT1H6wG8DIWOvVLu4TylOyO9n2HFPb2jQTwAzRy9zfDXu0Em4L0qAqc8c+pVvStNBTtFoJE7+S8wPnpfBL5ZCjI8s9R6PcGySL2fCTe94j1DvsnRxj2opoq9FgKSvNs1vD1iHhm+jedcPE+IPL0uiq+9Mif5vUJjrr133UK9raotO+Cupzy21v085hwevrrAjr3k4y+8o/KZPcXa8b1NDiY9FDIJPbKdY71yc8c9S7m/PdKjRb2hijI9fIqMvQrkBL0eAVy9fg6iPZjfwT0BxAI7GaUwvk5XFz7cGDu+OKnAvTKi3bxiEK68gQXJu/8ZjDy7zO+80f7EPcGMtryERJ89Ufe5vdEk073keHk9v4ufPXCMz71j5yC5PvKyvE7LTD2Wqry61bJZve3tdb3Brki9vVfEvQO7nz1BlbE97kGLvHbCr7zkYFO8uSj8vOGLpjxZpBu+Qu88vZCynL2FMgC9hOjoPaoH9T1tO4e64X4Hvq7WFz1Wc288OQqQvGyjyjraIzW6WUeQPRgHiTwFfT29xxOevWcIfz1zJ4M9NlDYvVY0fb67Dg69R0FhPeEzQzz11+U8UqG6vbto6j0ljLa97COavDuuKj0r2Uq9Opx0vUBYJr0GoTC9IIuDO9PXGTq14bY8jiPjvdePrb1lc2q6A/v/PZI9bL2b9XU8gu6gvWs0hLxebmE86YQzPfsCqDzBBJS9mVF/PP8FaT0XUpu94cpvu26z6b1umjg6fClYvR+W/bw+gCK9nHtAvRZ11rzkEwa+XHnaPemZvD36UNQ8GODDOx8rBj1yYlM9PjfCvYaWLr10Qlq9IldpPFycGT0adNQ8eST0vM6IrT1/Vk899hHvvN3kB72oe6q95mdUPUhGZL0FaDm+FUaMPBMIj7sZarK8h9yUPE92IDzyGN89xUOJPVExeTxeiBQ9n4lTPVqd/j3gA5y9uc+QPf1v2Lzsb7a8LkEcPsHoKz75L1O9MlLMPXlxuT0tDiK+ELcPPmGZSzuCfui99I65vaIQmj1ejKY918KIu3z31DyaOei8Lwztu5o+Gj5H9JQ9fYttPU7zGr6dvRa851XTOmK6Bj3qaAq8Aqo8PndYyb2/lIO9OOJmvBCwhb0cDyG9NgkCvmhN4D3pGWG9OuIiPXCoKT20U6G9x9EpvRxQNL2RuaE8SwqAvaqIAT1XjS69PHIaPJm5or2nRb89udurvcCBqbw63p882W3CO1ukqb1cyie9RaYNvDtRsbybHgQ8bAukPLUwWrwHrHM9C/pSO5sPyjufq4Y9pGVKvJVCkz3VTz89oQHWveEF2T1tMci9f91vveR0E75zjMI8/2eUPA7SubukPR68J4yfPZx0E7sS2Bc+tuc/vUd18L2cNRo+PEPgPOmkOjxBsDw9g97ovFEnNj1JcCw9WrG9vaOqATuKR349dhGvvb39Dj4uwCY9y/FWvanQaDzhz8Y9KHbWvYhX3TwZjx++a3ELvkkWAL1U+5C806ESPoBCCz66lAM9vvbLvYN/JL3RCBA9pBDBvRvfUjyr67o8pVOePFkLHr0V2z+9Wj7cujx7xT1yTQI9B6M0vWqTI77l0pu8vuc7Pf+Ukjtz/fI8k7VMvebjzz1UMBe+lRSyPCO5ej1NU7e9XZYuvUbDbT2541y9Q4/FPDMM0rk0o/26V9KovUqI1ryoBA49Af/kPXOeOj2DqMI82ZJgPDTrfb0ty7M9AcrjPF8h2D3CYsc8ekEPPDHYwz0Kf7+9E8SROgWfWr14K6E9IPjQu73bAr1G2Je9GeiZu+cHL73NkAu+dCNVPS8Mzz2CBGk9OVMEvC7obT1iwZW8BSmnvOnljb0Yr4W92etYPN2Fdz0g+QU+fjdBvQRejz35m589XxmkPObrBr3/WLG882nmPePJ8bt74628gZ0RPZOrKD2dZ2O9J74GPRCfubvJsGa9gw0XPKbuaj2PbGK9PrJ0PYXrAT43mxi93oUePflmEL1rara9DhjaPYKf7T0L0iu9OeXsPXVn6T2xAIm9KcyAvJLZFr3Avi68dQILvoLlgr3ZbOQ6yeNZvfaHlj3BBYi9oQRxvZjH2T0Uv809aMpIPel69r2V9Sw7NgO7vCRmrjygY3i87PgUPml9h72O74u9cleBvBnmoLxjEIq8/eQWvoK1g7v1U5S8u4CfvGjxwz0yy9C9oh9JO8gv5L0cZ0g9w3+nveAGjr0CJ8S9LMG5vCgShb0v6LU9ZUPPvTfVNr1vYuI8Td/ivCfSpb3lkHc91f6YvS8mtTwXSAU9QHRUPagKGrw+Jzk9vagDPRa/krwUDrw856psvJmGBj6PXLA67qASvkgayz09I+K8ERstvA7hPb49pyq78rVOvQzKqLuHZZO9Et1gPb/oOz3+89o95uTWvTK8YL44shs+bVnVO6aQnr2tVZO8oXbAvCM81LwtDD099TQOvX59bzkhSeC8ESOkvUZ2Lj4DX849TAQBvYWkn73ZmJg8JtfmPBqEtjvm4ku+e1hIvUO93L1mAy47mOYdPhsJDz6hsnO8uzgGvrcGGL1OejE8X7jsvFTrRj1G90E7ap+WPb3DJD3mcdG9wLBxvHrf4zrJ0oS91d2hvYRw1b0dnzS6BjFfPIuD3rohLBQ9N9cFvRBgDT0TR+i9BXDPOxfbhb1LBxC9lCalvfDQ1rulFXu9fP8ePYBaAL0VveW7Kh+wPFEfq71aqEM9BKnFPcQFmL0BnbO8sxaNPMJNw7uNcaQ8QEHgOzXHrT0v9jk9yd+RPADXtT0GCti9jm33PBs2AL4xh/Q9A1cwPTlSN7y7dOK9G3C5vGMcLzyt7du9r2LLPfppP7td7iW8GgIkvLo6sjsY/7s8J0ADvnwgm70nhvu9KckYPWt0JT3vo0E9UsFavSlx6j1rTPY9eo0dvb85JzoGBYE9tjQ0Pdw8CbwRfEK9V19zPOHarz3zyMs8CQLmPGWHgbwnAn0928YpPWPb0D3mqrS8PY31PLiIjT0a4vq9ka6sPae9Fz2p/Ni9mp27PS2g0TyWpnO96YEZPjhcDz6wqmC9znIHPs52STzZkei99gqdvSoJ/rwFR6s9Q1K1vOs4gTyQmU28H4byvN+hYT2AD9I9trTqPc8SFL5Gtl48YVTrPCK34Dwtxx29RYwZPoVO7739jwm9m9OSPOe0570YlsO8DMM8vvAYHD2hdbM6YfeYPG7L2j35Nsu9zOoEvucFvr0j3bm9uXaQvKoeYb27qqC9lP0ovM27E73Ryo097RSmvSZ9h71JdAq8KUetPBqkgL1o8pI8+SeMvNlgeLyVyHE9qlgXvL8ldr1Twls8nh3uvXdcq70WIPu8BiKBO9aSnj2xy7w9n+sFvg7/IT7Qjpy8JNZ1vTnKq71LMbC8LM72vI3xez17G5y8HqbkPXMDkT0cQ4I9G6ynvafgRr05u4c9IJjhPRhKSb2P4Rc9ugY3PWbETT2To4g9iaStvZMnI734GzO9MVEWvdtfQz4j/hE+mwF+vTvueDySD6o9yzk6vTp0jTz78Ai+UeF7vScMBb4v2b88EsQcPkq37j1ok9u7c7ePvYmqYLupuhM9cVonvQVtozu+y208rOCFOyHjGb14iGC9qBCPvNkuUT2Mev+9xH+UvS5RBr5XrXu9vaPzO/ayozyDe/U8KYGMvSq8CD0OOzG+jpx0vXNGLbxiHZO9PPhfvTPQbj19z369Ik9bvDIQqjxA75w8ReYgvBvOcL2t75Y9iJwBPkgg7DoTPH291BqvvDPzsrtG3Hc9xW5qPf8toztwpP87kuTbPVy00zzMum295LyDPSyK4L21HQg8w1m3uyBFIj316zy9KN1+vQlNiL0sC9u9EYmEPUV5BT5os/08Y5aJvf6X5z220A89nzcCvv6hFrxcuZ29m/S4O2eKOTwg5xC8mJeCu2Xx5j2dsxs9oRl7vapRBD1kI7u8bGakPL5GvLzj5Nm6EPV6vIpD6ztNzXG9zFH5PO5Srbx8bQ4+0QApPaIbwDxgWko95tlcPXMSDj1kr5W8+pAJPiwZHD0qwsy9KxVBPZuEpz1EDAK8POzRvHEmUD2vg769R8ObPKYzLTyotow8FNDIvRFxij2agD299uREPf9/u7wjZ0W9CBBYPRAf7j0EP9c8vmtXPTICLr1slXW8iA27PIB2Az7ryiq8C54GPrpRG74lBE49iZ6fvOhf+zwxWRm9COkUvnfy/D1DQUI9gKuiPBEFTz070W28iNwMvlBX6j3+xYa9kz3QvaS89zyIdpS8wkDuPNmrijvdPsm8kQgIvsJevbs2tya9QAjPPc1Yb7s7wAY8pyyaPeTwwD0AbBM9x62dO4PFeL19kW89v7SYvDfZqbyS6sq9OC/dvLoE5DzrELK7TeCsvHoilz2bn968nqFAvFgUR70ALxA+iPVxPGPjKr3liGi9Te5MPPabrrqNBng9SZhmvV79xbxI4yk+fD7GPT69gr3mDVq8enMGPammHDs2tPY8oxlbvhHDvr2FTsw7xUuyvTuJ2z1kC6I8jQTJvXnKoDxST8U9IRluPdObtD0Imrg8ACwbvsCmR70nuey808nuPb3ICj6aoJu93IIbvRXh3b3eT4k9XMOLPeoZojxvdwa9rcXAvF34srxzWSc9XnWjvdxOAz5tWog6ec0LvmLg1723Fr28NM/fO7JqgT2mOY89K1ywvKkVOz4RoVI8Bq/CPUEyIj3DP8a8pSaBvBxpdDwxSQy9JazIPebWhr2emZo8BH8UvrQEcr3E8xk9PyS2PX0Wlb0rXl6995IUPW7tGz33RjI8+bFqPfgQ0z1ycsC9O6GVPRthfz0Jxao8WgYZPC9fLr7Q4Yk85trivTYv2zspG7M95q2hPZ/uAjzRSKU7biTuPfddhzwUWtE8lnJXPFmBaTzplgE+tkAJvhJ/ZL1OzMW80m1GPLSMpDzOpw8+qkLcvUeNqj3OGpw8HTYbPWUknTzB8iS9fnb2PRMhxb0ZHMy8whTbvK4nRzwpWQk6NYYYuzUnMr0h5SE+Dd8+PUyJWD1KpSc8/7s3vC8fCT6JA7q9SWlRPDCmHj3TBpm9JKtVPfqjwjx3LGk8DqloPQbx4Dx0N0W9JaH/O22yxryLBLS9rpWDvb94AD1trhQ9XSgUvcKb87wKqPU6JRTjPEJSDT74FCk+UWngPKPBoL1PFBw9j0AfvETGkT3bDOc9iVcOPuZwNL4cqt08kqFou6iQRb3xZK+9boPmvb0llT0XWuu82xmJvRvJ8TyCYuK9TRbKvfHhAT11fkC9Xo22vYfRgbymypO9oXxvPf1e0L1zBNC9Y99mve2zCz1d87q9bpuyPXdUED5ZOf07qMOsu8T9az1/fzk8GhJ1PHLWmbv6w1U8a7jHvY0ugL3+m/C9hP5/PA84Vj3LfNE9yfa2vQZDkzy8Z6e9BPYTPXueUb2k6+g96GRhPRH6tr2v5uK8T7WAPcHWIr31nEA8v757vWy6e71gJrE9Z7XsPexP7L0glDU9PxiavLbXyLz1fV689A35vZPh8713ASS9rsUSvnyhvj1Fbxs9CULoukrJzbxJ5j89iqMHPZzatD1rTJG9d08LvkTVkL17ixu9NaClPdWWAj7h9/Y6OS67vcwx4LsnrrO84fUPvdJDVz1xo6I74gNmvWiPtrygkTE9ZP0CvkN71z3mZPM7J3fdvbVCGb5P8mu9m40CvQiADz0PjrK8TcnZuouUMT5aIZO9yiNePMYxir0BuRW6mrDCPK8EozzW8vS9fMECPUWqHr0wvZO9Ovfjvah7Prxf9ZA9JNf8PQeZTL1G/Om8TNduva+tcz1hFw69VMWkPaVdJj4b2x88Mht7PdJ2ID0rlp87e+fNvHiYqr0DzFk9r8JOvbK9tj1NIdo9bRn3PH8j97xf2aK9hxnjPYMH6bqKIQk9N8MhPWZSCT34nYo9myscvubBsb3Zsgq+1NDjuk4+VD3LVaw9bl0pvdwvGT0RVLG9bCPbPF0oeD1Bchq8VHBmPcl7AzzM8I69mGqJvXb0Izx//L28FqNcvf3UFD04RAk+mMECvAaLED3PrJe6ERMsPLGaJz0agQO9UBhtPRepKj1NctO9e/tvPIR2CD2w7Pg7lw0ivbrc0jyp72a9hxGUOwSLvDu/Epa97dfFvaT1qD337xi9YCrhPMtEDz2o+JI8GeeVPVuk1j17jh891F2OvN0Kcb09Vbk9vTajvVkVCT5trT+8Jf4/PtbOG76+cs08WmhYPXksTr0Xg6u9czMUvid1hj0+ZIi9SI+VvP1LpD1uWGq93kbDvewMtj3x9PG9x80Dvs6HJrw9fCG96lvdPf45w7xPlx486M75vTQL3jx8QTK9gWkFPj0WLL1Ul8K8Ii4RuvyX4T2v8G08XjiTPat4+bxiGQE9XyfNvXh/Jb3hy4e9DsCVvOSJvLw5b6u9G2VUvZA7AT44Ht+9MZ+2vHzNML3mye09zMlfvftOAL3cMLS8b53APF/kiTzOzlQ9kZyZvBDa2jwVoug9FieKPSyoLr7xpMk8FqNpvfSsCDyg5v09VHcnvkWB3719xwi9rhD7vD4dLT6gQU68xQzCvW/N0LwOWJM9Hfr3POXJPT3oqMy92pXZvZaNrb0ujKy8BbjZPWenID6W+/U89AzcvJIEfb2xpvg8XQ9BPUlMqzwnJPO8PGHevLBhi7375I49ONgAvkSymT120E09uyuGvQ9iwr1DYUe8ofuRPCvluTuLcV89/G7HPMUNFT6R2TS9cAtYPR+9qzvTHuC88kf/PBRGhDy8HY29udwAPo5qPjuTsvS8qlBkO8sjDb1NA0g9QMMAPti9Db132xS9ExYQPSrnYzzbAF+9udPUOsKx0j1RTSU9uqmPPZGrBzzg4Nm8aODQvLaXRb716kg900aNvei1pLwRGnM9c9ySPWIZWD0uOxI9WnnyPIogoj1v+uU8itIiPT24Zj0H9Dg+eMCjvYbefjwR+gi+IHb0u6dQNjxEyvE94RSwvSBquTyinum8qVOgPPpHND1d7Im9cwSgPRBgN71FK4S9r0kVvlzEFTyDvGc95bQivLDVHjxE1BM+aAgIui1WRD3AOnO979CPPUHjtD0j+gm98B9HvQ10IrxTL228d6vTPQ0Y/jzkU5e9cI6KvTGTMb12sYy9DV4LvE4WZD15QCe+CYjDvUPbuj37Ssw7Irk5vD/YAz1WSNY7OhHZPW0pmj3mzFk9FvFuPetF/rzdgWU9fQ5Vve36oz1ui007WBQaPlun972LQIE9Sz6CPPU7mbwfydS9QYMDvmHEgj28Mta9tzZjO1ZSoDuaF2e6jWr4vavo/D2FUsi9W+SCvfBcOL15s868N2wcvNkYtLzZbEk8dnjYvWi0ortlF286luXSPeoC0TtXvzS8M14FvcBlBT5WGM865IxvPQlhRDzkUW49xSaFvaC2i7yYoYy9qBDXPNdokjwRvfO8uICru1LkpD0Zw0m+RUTEPOTN/LuFkRE+L3MUvX03Er3i5YW9JhiPPdiEyD3KpBo9V/jrOn9QTL2L7DU+uEtlPYN7G74TXQA9bxGrPNHtiDw2rQU+WWAhvvd0wL0Q9l88NLmJvRYZKj3d/nE9qFAGvsoAgDxE48U8kCsHPUMOVz1x/IC9gdgZvnkYgb00wJo80lDUPZIe9jwv+OC7wMycvU3pBr2Bjoc8GdtSPcGOKj1tCMi8Q2divVNbAr0LhW89ogQ5vtEWLD1AxUw8GXPDvSkFW77rUJI8Y19jPWzlDr2ZjBY60HESPRyLkz07w/G9ybUEPh8tYj3Iyp26hGAIvYrTmj2A2ya9MaMBPnItkr0gN6A7oy1WvUKNpbttAPA8O13XPe5kSzzhpTi9pWjbPAFRjT2gZMM8Jjc8PQMi/z23CN08+EeQPVii6rw77TQ994jbvAeYBb5WjMA9E4OXvfCvITz1OgM9UPupPYr3pbwnfHe9Cwn5vFZEPD3yGGk9EWaGPNgs2jwS33o9JE4ZvvRJwbyXrg+9AOiEPdzNhDvbv+o9oRbAvbTyRLzmOJK9UvwtPdETBz1O7xu8gPstPXV11Dz9HgG+pJ3gvTFsHD2/1TW88anyu6yjtbuyfv49bnxBPXiyNTwk3qU8W23CPNKYNz1vyzq91WsxPeNl9ry/WX28H7SHukBgMz00FWS97DjdPaYE9TzFYvO8iH3wuuCB2rsSV6u9NPBjvSkOoT2IGts9i8ikPIxMoDzs9Sk5WbrjuR+ETz3Qgjk9OIRBPTslf73NMny8YhWPvT68hj1T6Qs9vbXyPWDiWr7sdtU6nHRIPO+7mr1qPlq9NprCvUo5kz3PZ5W9NW3COz0KuDw4MKK8w9oFvmN7RT1HJaq9+MNsvQCJmDvH/R+94lutPcFPijzqYkS9V4LKvYo8x7xvjk+9WxW/PSG5Jj0IuKk8qCa+PeK04j0U1BM9CLWbPez+Cb1gfK08MfbbvDW3TL0qF6S9kQzxPBRnkz0kqhO94MKnvdV/oz3h2hC+5DURvI9bZ725NAg+Ql2KvSv0n73PmqW9OhQVPUPfIz3FYJA9wSVQvC43KL1/ZP097FzIPWTA3b0Syss8QHVUugb3yzywf7s9wXcjvv0SDr676RG8uwnzu9t8Az4/oJo92aewvVhsvjxLU6C8a+RbO28DHz0w6Vy9IKUWvs+Ds72KQPG8UgDbPeBeCD4nbAG9PIzbvZHH0LzzDZ28w2C1PFQy1LtXhVm9meHjvKtzpb0GGac9MXH0vVS+WD0QH149v6CNvc9tMr5+DL+8rjN8O7G2GT2OGiE8T4lKPSOUHT5Cd1W9S5nFPQNq5jy9SqS9VlgEvSXUAz5mq9u97/gZPTLhnjrTB8e7ZGTfvdL23rzWoIo9ceImPgaXDLwSo2e9qCbDPOYMrT31sQe9spJ9Ou1NoD3bOUE8m9rcPUl4uTz1aX86G2vJvSUZU74V5yE9F+K9vVGwZbw/NDQ9wKsmPVaKeDxwkD88H8uoPGtspz1qBhM9uD8lPbUmoT2KcuQ9zX0Zvj1nib3AeRG9+mdIPSOsY73jPPw9qZucvTz0jD1A9g08GAUZPKOFiT1IZkC8II+FPYv7fLxR9pm9DSksvv3/UD07/gI7aCqzvNhVEbxczms+qdEDvFUetj1lYDs944LAuzlL0D3uTOm9SyMVPSG7lD1o/ms9w3PBvBtNKbzGV7M80aNAvFEoKT2x4gK9ieQtvagsZT0txgK+whE5vfv4hT0EQ/a8f4GAvHvsCz2S/+u8EyqUOy46Az5xxKw9DjYCvR3b6LyPT0I81l56vVcXLT5HZNU99smyPetKHb5NMdc8dNp5PGGFa73UIa48S9qFvRxNoD2Z6Q68t8mNvSdqK7wDNu+8tzzHvV9Xk7zDp7C9kxaIvOtp6rsP5rq9aTtxPO7kar1Y5Cu97PbovSOPDT2uqQC9qWDuPcutvDzJ4YA8GRzLPKi/FD6kIO08fXyNPfOcyrpaHd07n0UkvrGdcLypY7m8nGFSPcy8YzyeEHK9uX7YvQceEbx/6AS+Y+NGPdXyQ73nPvs9/OhjPDGz/bwuD6m96yLKuhypBz2+dp098ni6vL+Onb3MQbc9FJwIPlv68r3Kg1Y9S7qmvP0QmD1G5vW8P6qYvUjtZ739nz+6AbY/vQD9FD7gIYI9Z9YmvXyaarysC3Q9/MO4PXE6wzwFlwC9TxkRvjXlpb0+Gyc9bUcRPb0MNj4KdD68jKmnvWBXIL2Z4Pu8qlWiPfdfPD14pFc9oQlzvVg6ob2LQWU9uNKpvREHmD07rYA9bJCcOQ6VHL5roJ+9TaQQPQT53DyldFi8LLZEO5FiQT60+Lu9Q2fQPA66srzMnqe9hXcEPUfXMrxWK9S9PNG+PL3OezwhrQo9/BQMveGPzrwXoOE88QtfPY/ljL3QYWe99kyvPIEAyj382oS8yBqRvZW33j3JiW686NW0PZXRSz1mDUa9n6cFvuWjDb7yqKg9M2e2vJqG9zvpfW87RUrvO8hi8DxuY1W9KTScvMeIybrmxrE9kjXlPDePBz7wYao9WzcevoJtZr3Sita9nUxjvEQxwjx1vvo9bKxqvXx+Kr3cPJi9hewFPO/LBLwtvYq9HVf4PPa2oD2Djam99LUvvq18pLx6DQe9UKMIvQUgoLxu9b49AkcrPT/ICL2l3Q08ICRXPSqK6T3yahu+2gCuPR2CojyKV7e855qIPe64Yj2fo4O9BLBUPDooXD2jfVq9OALGPPl7jjym4Iu9lMuMvQEvSj31Y309kDLjvIX0Dr0x/z66cwqhOzNCgD0Jiv88c58fPdhom712Gpi9muY+veIejT3c97A8FP0PPnJT7r2r0gI9nFxKvHeO2L1QuLa9BYSIvUMWuz3mXIm95EM+PLbRIjzcRcm8hQEKvnHEOzwnYmi9phZwvYFAhDzBcmQ9G5ybPfff/bsowNY8b6nhvZL6KryRkZu8csDQPFbTgj29N449lfyGPUwqzz26R8s9ELyLPTzNrL35I4Q9l2jxvA/a/r1DuBq9K2Yjvcb32DxIkiK6H0TZvdytpD1VGR++EMsIvLrDA72ug/g9KHgqvTCqXrwhSxS9prWBPf98iz2ODWA9OstQvb07Cj0jbA0+6IOCPTf/Hb5JppQ7s2LYPKvm5jxwX8Y9z9hBvrDy0jsg26q6KbfpvUhTiD3FvBc+wv+BPfzUhT02QLI8v/YxPTZRkjwbw9e9uNYjvevIrr3KbTm9uj7gPAjGaT1L1448gIaDvQ1GPj2bil49KJegvPaf5DyCqji89estvNh8Uz3/OCS9VLAIvgWStjxSUCe9R3JnvddvWr6WYf697WCaPaxibL1gKKo984c9PcBtoz25VCi9ZMq4Pb8XkzxVZqE8+3usvWkmJjzIPtW9R4j1PdqOyrzeiPw8pe9uO6fxnbw8/vo8S+3aPfCAer3mH8q9+qCuPYslAT0tIs47QunevDBFKj4AYM89KFdXPWgauj3alDa9pJeGvAWgAL7pprs9NvNGvTICjrxK9cA82K+EvaaWory5zqi9feqcvMoBHT1e9yY9bNhgvM6uJT19iiU+6RMkvnI1g7wbJMu9KKcsvdNuW7yeJbQ9avEnvbx51jwdjbI8g+1wPYMADj2mLGK6y1kbPuwH8Tw8tx2+CFndvfATGr2XhZu9zeeBvP40PrvsfIE9g43TO+zC2bzV7QQ7YjoRvdIkqz1JsY28hhgTvG9P9Tz+nz69A5KKPQ9ubj1r5ow8s/c0PRUPq71tD6G8akXOPFnOBr4q9sC9iagjvlDLjzyaDMQ9xqoTvFp07z004iA9UN3wPcBX2j2iHXk8mb//O7T1pL123je8TnvxOpZZOj0aZAQ9cQIjPqleQ72GdMI8jsNLPTuT+LyKOja8mLjKvVDGmD1T2Um9aTOhPcGvJT2QtHq92KWpvYxmDbsaIMG8KJZnvUDEyjyDh0m9AX0TPRo3xL2xAdG78lmNvFBzNjwSRqa8Wx0pPqx6tz0sAoG8e3z8PVVqBz4z3oM9oOm6vbngA73tquY8YAeyvbWtnr1ApZK9OiHkPATyGbxfKWQ9QVUcus67qL0ptTS9rpS0vRjTnb0ZEJc9SeRvPVBp+r0utwC82qSRu49lSj1HRZU9h2b/vAOMxb30+eo80f70u9cxh701RQq+uyKYvcJfBr2GKIQ916f1vXl7n7049By8USIXvgJKXT6rWSu94augvQFjPLvQPCG97j6qPb4vWTvzLQ29yszfvZrrBL53w4u9Dc4jPsHTUDxeb6Q9BSvAvWirDz5Ipda9WSeavUc4qjuFmrG9UD/hvSxvET3vQy49O+OYvdCqlT06n6Y9iBeevWjjQb2Wgki9XCrmvFLqqb0ueIi9ez+GvMStDj7RjT+9YT5YPIq+Db2xncq8+iX3vVh+MzzCZAi+jEKsPcVGsbzZvCq86+B/vN4lGb3qd6W7jHUmPtQCfz3SlEi94GYKvD1rID3dWaE8djiLPXk/tz0ADvg8Gj0zPdMK2j2hxKo8XlsgvW3wS73lzwG8Jkt4vQz7tj2kIgy98ferPI61SrzfSDg9EHPqPSzLfz1tU1c9MiCKPO7ljD0XHg09+Vf5vZsV+L3enWq9wUHUPOOrwr0OTgY+jkBnPTSHtD1FkGK99X0FvMrHqD2NMHS8oFGcvSd8lb2/KMu8SOIgvSNenD3HArS8Jw2zvSIQRr1dINs9VTEJPiYWmLxzJxa+q1ImPTscLzyVy807ZnJzO+Dx27yujYu9rCGfPQ2X9DyVGia9yRDLvTVC7T3+LYu9WWX3vOCyrz0Aa7q9tc7KO8EsID5vZko9vP2pPPEgRTuRgNo8VQa+PZjKMT7rezS9G0NUPbhFJ74A46s9C3lrvdBkBLxXOLi85zfnuv/ZCL46q+e7qSYsvLb3lz3GvFi7sFZIvha+P7zQHYM7sALvvdOJhD3/rf29b3mBPQP6kT0AEIG9eDeyvbn3tzq1B2W8N2ByPZsn1L2zicq8FsP+vX4Na73ie+29zEHyPV5vtL0tYN+9C3UfPSMUTTxqTJm9afJMtzuHCr3IDHQ9DUa1vF5bmLsyM5e8ir0NvVQwXT3y9Dq9vSkOvQ1upj2quPm92aoAvNLE3bwCGJg9KfH7PE3sFL51HX29JP4GPSgRYT0i9cC89yBZvX1Frr3t3gM+XMuTPFbPaL1oZK+9Dh2aOxq5hj2NVuI9IXGzvahsQrzzkkY9TtRGPKUAcj1gnq29KBS4vUtWb70KEzU9zn1UPFJugL0nyt68EVALvdQUML1hWeo8FIj2PU1NJz56rA29duY9PAjJxTy8Eic9r160PRGILb2I0pS8gxWfPM8W4ryW6M29hY8HvnpxxL3eHLc965++PAKnCLw0bwO9jYSEO12JXT1I4Zi8tIqTPacxwT1n9ay8kXpSPe0yVr3moM28IxQAvrmrMr0eJg69B9+8PXgzob1KFyo8OddovcZCQj3i7q09Bya6PR2OAD3MXgO+NVaDPA/SGry5edM9jjVwvB/SaryUsiO90w4BPh3CGj39CaU9LNtIvB1mX76M6N883tjnvTqZ7zxURHe8ekwxO3AijjwweZm9INa9PZ6KlL291wA8Ia2bPOVLnT38j1w92bPVvQ74oL3PDim9lsrPPP7n9j2GRJw9IoCovcaHxT2TaD+8utTfO3EF3rzpqJe90VpOPW1E6Tx9idW7RLqTvYnwsr0nJPI6bRYpPfBDzbw='}
reference_hashes = {}
for name, value in REFERENCE_FILES.items():
    content = base64.b64decode(value)
    (REFERENCE/name).write_bytes(content)
    reference_hashes[name] = hashlib.sha256(content).hexdigest()
enrollment_candidates = (list(Path('/kaggle/input').rglob('auditor_enrollment.wav'))
                         if ON_KAGGLE else ['references/auditor-reviewed-v14/auditor_enrollment.wav'])
enrollment_candidates = [Path(path) for path in enrollment_candidates if Path(path).is_file()]
if not enrollment_candidates:
    raise RuntimeError('The attached Kaggle dataset must contain auditor_enrollment.wav')
enrollment_hashes = {hashlib.sha256(path.read_bytes()).hexdigest(): path
                     for path in enrollment_candidates}
if len(enrollment_hashes) > 1:
    raise RuntimeError('Found multiple different auditor_enrollment.wav files in attached datasets')
enrollment_source = next(iter(enrollment_hashes.values()))
shutil.copy2(enrollment_source, REFERENCE/'auditor_enrollment.wav')
reference_hashes['auditor_enrollment.wav'] = next(iter(enrollment_hashes))
(RESULTS/'reference-hashes.json').write_text(json.dumps(reference_hashes, indent=2))
print('Reference bundle ready:', reference_hashes)


## Credentials, checkpoint restore, and attached video

Create a private Kaggle secret named `HF_TOKEN`. The token is read from the environment and is never embedded or printed. The current video is copied from the attached dataset because YouTube blocks Kaggle's shared addresses.


In [ ]:
if not ON_KAGGLE:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A Hugging Face token with diarization-model access is required.')
ENV['HUGGING_FACE_HUB_TOKEN'] = ENV['HF_TOKEN']
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
    mossformer_cache = CACHE/'mossformer2-items'
    for archive in Path('/kaggle/input').rglob('mossformer2-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (mossformer_cache/member.filename).resolve()
                if not target.is_relative_to(mossformer_cache.resolve()):
                    raise RuntimeError('Unexpected MossFormer2 checkpoint archive path')
            zipped.extractall(mossformer_cache)
        print('Restored per-window MossFormer2 checkpoints:', archive)
    overlap_cache = CACHE/'overlap-extraction-items'
    for archive in Path('/kaggle/input').rglob('overlap-extraction-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (overlap_cache/member.filename).resolve()
                if not target.is_relative_to(overlap_cache.resolve()):
                    raise RuntimeError('Unexpected overlap checkpoint archive path')
            zipped.extractall(overlap_cache)
        print('Restored per-window overlap checkpoints:', archive)
    expanded_checkpoints = [path for path in Path('/kaggle/input').rglob(VIDEO_ID)
                            if path.is_dir() and path.parent.name == 'stage-cache']
    if len(expanded_checkpoints) > 1:
        raise RuntimeError('Found more than one expanded checkpoint dataset for this video')
    if expanded_checkpoints:
        shutil.copytree(expanded_checkpoints[0], CACHE, dirs_exist_ok=True)
        print('Restored expanded stage checkpoints:', expanded_checkpoints[0])
OVERLAP_POLICY = None
policy_matches = []
search_root = Path('/kaggle/input') if ON_KAGGLE else Path.cwd()
for head_path in search_root.rglob('head-to-head.json'):
    try:
        head = json.loads(head_path.read_text())
    except Exception:
        continue
    candidate = head_path.parent/'overlap-review-policy.json'
    if (head.get('video_id') == VIDEO_ID
            and head.get('revision') == 'sortformer-additive-two-tier-policy-v5'
            and candidate.is_file()):
        policy_matches.append(candidate)
# Kaggle normally expands dataset archives, but accept a retained ZIP too.
for archive in search_root.rglob('*.zip'):
    try:
        with zipfile.ZipFile(archive) as zipped:
            names = set(zipped.namelist())
            for head_name in [name for name in names if name.endswith('head-to-head.json')]:
                head = json.loads(zipped.read(head_name))
                policy_name = str(Path(head_name).parent/'overlap-review-policy.json')
                if (head.get('video_id') == VIDEO_ID
                        and head.get('revision') == 'sortformer-additive-two-tier-policy-v5'
                        and policy_name in names):
                    extracted = WORK/'attached-overlap-review-policy.json'
                    extracted.write_bytes(zipped.read(policy_name))
                    policy_matches.append(extracted)
    except (zipfile.BadZipFile, KeyError, json.JSONDecodeError):
        continue
unique_policies = []
for candidate in policy_matches:
    if not any(candidate.read_bytes() == existing.read_bytes() for existing in unique_policies):
        unique_policies.append(candidate)
if len(unique_policies) == 1:
    OVERLAP_POLICY = unique_policies[0]
    print('Found required additive Sortformer policy:', OVERLAP_POLICY)
elif len(unique_policies) > 1:
    raise RuntimeError('Found multiple different matching v5 overlap policies')
elif REQUIRE_OVERLAP_POLICY:
    raise RuntimeError(
        'Attach the Kaggle dataset created from sortformer-comparison-results(4).zip. '
        'No matching v5 overlap policy was found; stopping before the full run.')
if not VIDEO.exists():
    if ON_KAGGLE:
        accepted_names = {VIDEO_ID+'.mp4', VIDEO_ID+'_full480.mp4'}
        matches = [path for path in Path('/kaggle/input').rglob('*.mp4')
                   if path.name in accepted_names]
        if len(matches) != 1:
            raise RuntimeError(
                'Attach a Kaggle dataset containing exactly one of '
                f'{sorted(accepted_names)}. YouTube blocks downloads from Kaggle.')
        source_video = matches[0]
    else:
        local_video = Path.cwd()/(VIDEO_ID+'.mp4')
        if not local_video.is_file():
            raise RuntimeError(f'Missing local video: {{local_video}}')
        source_video = local_video
    # The supplied YouTube video uses AV1, which Kaggle's OpenCV build cannot
    # decode. Normalize it once so the full visual pass actually reads frames.
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-i', str(source_video), '-map', '0:v:0', '-map', '0:a:0',
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '22',
             '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '160k', str(VIDEO)])
video_codec = subprocess.check_output(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=codec_name', '-of', 'default=nw=1:nk=1', str(VIDEO)],
    env=ENV, text=True).strip()
if video_codec != 'h264':
    raise RuntimeError(f'Expected normalized H.264 video, found {{video_codec!r}}')
print('Normalized video codec:', video_codec)
print('Credentials configured; token not displayed.')
print('Video ready:', VIDEO, VIDEO.stat().st_size, 'bytes')
(RESULTS/'run-input.json').write_text(json.dumps({
    'video_url': VIDEO_URL,
    'video_id': VIDEO_ID,
    'normalized_video_sha256': hashlib.sha256(VIDEO.read_bytes()).hexdigest(),
    'notebook_revision': NOTEBOOK_REVISION,
}, indent=2))


In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', BASE,
                            str(CACHE.relative_to(BASE)))

def stream(command, log_name, failure):
    log_path = RESULTS/log_name
    recent_lines = []
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
            log.write(line); log.flush()
            recent_lines.append(line.rstrip())
            recent_lines = recent_lines[-20:]
            print(line if len(line) < 1000 else line[:1000]+' ... [full line saved]\n', end='')
        if process.wait() != 0:
            tail = '\n'.join(recent_lines)
            raise RuntimeError(f"{failure}\nLog: {log_path}\nLast output:\n{tail}")

def run_test(video, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(video),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--face-priors', str(REFERENCE/'face_embeddings.npy'),
        '--output', str(output), '--cache-dir', str(CACHE), '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        stream(command, stem+'.log', 'Pipeline failed; see the saved log. Retry with BATCH_SIZE=1 for CUDA memory errors.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    text_repeat_candidates = result.get('text_repeat_candidates', [])
    (RESULTS/(stem+'_text_repeat_candidates.json')).write_text(
        json.dumps(text_repeat_candidates, indent=2)+'\n')
    text_repeat_rows = [
        f"[{row['left_start']:.2f}-{row['left_end']:.2f}] {row['left_text']}  <=>  "
        f"[{row['right_start']:.2f}-{row['right_end']:.2f}] {row['right_text']}  "
        f"(similarity={row['text_similarity']:.3f})"
        for row in text_repeat_candidates
    ]
    (RESULTS/(stem+'_text_repeat_candidates.txt')).write_text(
        '\n'.join(text_repeat_rows)+'\n')
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    repeat_rows = []
    for segment in result['segments']:
        for evidence in segment.get('evidence', []):
            if evidence.get('source') == 'repeated_presentation':
                d = evidence.get('details', {})
                repeat_rows.append(f"[{segment['start']:.2f}] {segment['text']}  <=>  [{d.get('donor_start', 0):.2f}] {d.get('donor_text', '')}")
    (RESULTS/(stem+'_repeat_candidates.txt')).write_text('\n'.join(repeat_rows)+'\n')
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Repeated-presentation groups:', len(result.get('repeated_presentations', [])))
    print('Short text-repeat review candidates:', len(text_repeat_candidates))
    return result

def extract_clip(start, duration, name):
    clip = WORK/name
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-ss', str(start), '-i', str(VIDEO), '-t', str(duration),
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-c:a', 'aac', str(clip)])
    return clip

def run_targeted_review():
    output_dir = RESULTS/'targeted-review'
    command = [PYTHON, str(WORK/'review_transcript_regions.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--target-reference', str(REFERENCE/'voice_embeddings.npy'),
        '--other-reference', 'Opening_officer='+str(REFERENCE/'opening_officer_reference.npy'),
        '--output-dir', str(output_dir), '--cache-dir', str(CACHE/'targeted-review'),
        '--weak-confidence', str(REVIEW_WEAK_CONFIDENCE), '--short-seconds', str(REVIEW_SHORT_SECONDS),
        '--minimum-gap', '5', '--context-seconds', '3', '--maximum-window-seconds', '30',
        '--window-overlap-seconds', '4', '--device', 'cuda' if ON_KAGGLE else 'auto']
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'targeted-review.log', 'Targeted review failed; see the saved log.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'comparison_summary.json').read_text())

def run_overlap_extraction():
    output_dir = RESULTS/'overlap-extraction'
    completed_report = output_dir/'report.json'
    if completed_report.is_file():
        saved = json.loads(completed_report.read_text())
        summary = saved.get('summary', {})
        if summary.get('selected') == summary.get('completed'):
            print('Reusing completed overlap extraction:', completed_report)
            return saved
    command = [PYTHON, str(WORK/'review_overlap_extraction.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--enrollment', str(REFERENCE/'auditor_enrollment.wav'),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(output_dir), '--device', 'cuda' if ON_KAGGLE else 'cpu',
        '--whisper-model', 'large-v2',
        '--cache-dir', str(CACHE/'overlap-extraction-items'),
        '--snapshot-archive', str(BASE/'overlap-extraction-checkpoints.zip'),
        '--snapshot-every', '5']
    if OVERLAP_POLICY is not None:
        command += ['--selection-policy', str(OVERLAP_POLICY)]
        print('Using additive Sortformer policy:', OVERLAP_POLICY)
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'overlap-extraction.log', 'Overlap extraction failed; baseline results remain valid.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'report.json').read_text())

def run_mossformer2_review():
    """Separate overlap candidates without modifying baseline text or identity."""
    output_dir = RESULTS/'mossformer2-review'
    inference_dir = output_dir/'inference'
    labels = output_dir/'selected-overlaps.json'
    output_dir.mkdir(parents=True, exist_ok=True)
    prepare = [PYTHON, str(WORK/'mossformer2_review_policy.py'), 'prepare',
        '--baseline', str(RESULTS/'full_video_evidence.json'), '--output', str(labels)]
    if OVERLAP_POLICY is not None:
        prepare += ['--selection-policy', str(OVERLAP_POLICY)]
    checked(prepare, cwd=WORK)
    command = [PYTHON, '-B', str(WORK/'run_mossformer2_separation_experiment.py'),
        '--video', str(VIDEO), '--labels', str(labels),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(inference_dir), '--context', '3',
        '--device', 'cuda' if ON_KAGGLE else 'cpu',
        '--hf-home', str(BASE/'huggingface-cache'),
        '--cache-dir', str(CACHE/'mossformer2-items'),
        '--snapshot-archive', str(BASE/'mossformer2-checkpoints.zip'),
        '--snapshot-every', '5']
    ENV['SPEECHBRAIN_CACHE'] = str(
        BASE/'speechbrain-cache'/'spkrec-ecapa-voxceleb')
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'mossformer2-review.log',
               'MossFormer2 review failed; baseline results remain valid.')
    finally:
        export_checkpoints()
    report = output_dir/'review-evidence.json'
    checked([PYTHON, str(WORK/'mossformer2_review_policy.py'), 'evaluate',
             '--report', str(inference_dir/'report.json'), '--output', str(report)],
            cwd=WORK)
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads(report.read_text())

def run_caption_gap_review():
    """Use optional YouTube timing evidence to find review-only transcript gaps."""
    output_dir = RESULTS/'caption-gap-review'
    status_path = RESULTS/'caption-gap-status.json'
    caption_dir = WORK/'youtube-captions'; caption_dir.mkdir(exist_ok=True)
    attached_roots = [Path('/kaggle/input')] if ON_KAGGLE else []
    captions = []
    for root in attached_roots:
        captions.extend(root.rglob(VIDEO_ID+'.en-orig.json3'))
        captions.extend(root.rglob(VIDEO_ID+'.en.json3'))
    output_template = caption_dir/(VIDEO_ID+'.%(ext)s')
    if not captions:
        command = [PYTHON, '-m', 'yt_dlp', '--skip-download', '--write-auto-subs',
            '--sub-langs', 'en-orig,en', '--sub-format', 'json3',
            '-o', str(output_template), VIDEO_URL]
        download = subprocess.run(command, cwd=WORK, env=ENV, text=True,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        (RESULTS/'caption-download.log').write_text(download.stdout)
        captions = sorted(caption_dir.glob(VIDEO_ID+'.en-orig.json3'))
        if not captions:
            captions = sorted(caption_dir.glob(VIDEO_ID+'.en.json3'))
    if not captions:
        status = {
            'status': 'captions_unavailable',
            'review_candidates': 0,
            'automatic_text_insertion': False,
            'speaker_identity_changed': False,
        }
        status_path.write_text(json.dumps(status, indent=2)+'\n')
        print('Caption gap review skipped: automatic English captions unavailable.')
        return status
    if output_dir.exists():
        shutil.rmtree(output_dir)
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    checked([PYTHON, str(WORK/'build_caption_gap_review.py'),
             '--captions', str(captions[0]),
             '--baseline', str(RESULTS/'full_video_evidence.json'),
             '--video', str(VIDEO), '--output-dir', str(output_dir)], cwd=WORK)
    manifest = json.loads((output_dir/'manifest.json').read_text())
    status = {
        'status': 'review_ready',
        'caption_type': 'youtube_automatic',
        'review_candidates': len(manifest),
        'nearby_transcript_duplicates_suppressed': True,
        'captions_do_not_identify_speakers': True,
        'automatic_text_insertion': False,
        'speaker_identity_changed': False,
    }
    status_path.write_text(json.dumps(status, indent=2)+'\n')
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return status

def run_diaper_overlap():
    """Run official DiaPer on full audio, then compare without changing baseline."""
    source = WORK/'vendor'/'DiaPer'
    checkpoint = source/'models'/'10attractors'/'SC_LibriSpeech_2spk_adapted1-10'/'models'/'checkpoint_100.tar'
    infer_config = source/'examples'/'infer_16k_10attractors.yaml'
    if not checkpoint.is_file():
        if source.exists():
            shutil.rmtree(source)
        source.parent.mkdir(parents=True, exist_ok=True)
        checked(['git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
                 'https://github.com/BUTSpeechFIT/DiaPer.git', str(source)])
        checked(['git', '-C', str(source), 'sparse-checkout', 'init', '--no-cone'])
        checked(['git', '-C', str(source), 'sparse-checkout', 'set',
                 '/diaper/', '/examples/infer_16k_10attractors.yaml',
                 '/models/10attractors/SC_LibriSpeech_2spk_adapted1-10/models/checkpoint_100.tar'])
        checked(['git', '-C', str(source), 'checkout'])
    # DiaPer relies on a small Perceiver change from the authors' Transformers
    # fork. Install it into a private overlay so the established pipeline keeps
    # its own dependency set.
    transformer_overlay = source/'python-overlay'
    if not (transformer_overlay/'transformers').is_dir():
        checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--target',
                 str(transformer_overlay),
                 'git+https://github.com/fnlandini/transformers.git@b830ec2245139b157576153cfd8999e1da24a82c'])
    # DiaPer imports only the Perceiver model and does not use tokenization.
    # Keep the host pipeline's current tokenizers build and disable only this
    # irrelevant upper-bound check inside DiaPer's private overlay.
    dependency_check = transformer_overlay/'transformers'/'dependency_versions_check.py'
    dependency_text = dependency_check.read_text()
    runtime_loop = 'for pkg in pkgs_to_check_at_runtime:\n'
    skip_marker = '    if pkg == "tokenizers":  # unused by DiaPer\n        continue\n'
    if skip_marker not in dependency_text:
        if runtime_loop not in dependency_text:
            raise RuntimeError('Could not patch DiaPer Transformers dependency checks')
        dependency_text = dependency_text.replace(
            runtime_loop, runtime_loop + skip_marker, 1)
    dependency_check.write_text(dependency_text)
    # The official 2023 script's GPU check treats GPU index 0 as CPU and asks
    # safe_gpu to allocate devices. Kaggle already assigned CUDA_VISIBLE_DEVICES,
    # so use that allocation directly.
    infer_script = source/'diaper'/'infer_single_file.py'
    infer_text = infer_script.read_text()
    infer_text = infer_text.replace(
        "if args.gpu >= 1:",
        "if args.gpu >= 0 and torch.cuda.is_available():")
    infer_text = infer_text.replace(
        "        safe_gpu.claim_gpus(nb_gpus=args.gpu)\n", "")
    infer_text = infer_text.replace(
        "librosa.get_duration(filename=filepath)", "sf.info(filepath).duration")
    infer_script.write_text(infer_text)
    models_script = source/'diaper'/'backend'/'models.py'
    models_text = models_script.read_text().replace(
        "map_location=args.device)", "map_location=args.device, weights_only=False)").replace(
        "map_location=device)", "map_location=device, weights_only=False)")
    models_script.write_text(models_text)
    # Librosa 0.10+ made mel-filter arguments keyword-only. Retain DiaPer's
    # published feature settings while adapting the call syntax.
    features_script = source/'diaper'/'common_utils'/'features.py'
    features_text = features_script.read_text()
    legacy_mel_call = 'librosa.filters.mel(sampling_rate, n_fft, feature_dim)'
    current_mel_call = 'librosa.filters.mel(sr=sampling_rate, n_fft=n_fft, n_mels=feature_dim)'
    if legacy_mel_call in features_text:
        features_text = features_text.replace(legacy_mel_call, current_mel_call)
    if current_mel_call not in features_text:
        raise RuntimeError('Could not patch DiaPer for the current Librosa API')
    features_script.write_text(features_text)

    audio_dir = RESULTS/'diaper-overlap'/'input'
    audio_dir.mkdir(parents=True, exist_ok=True)
    audio = audio_dir/'full-video.wav'
    if not audio.is_file():
        checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
                 '-i', str(VIDEO), '-vn', '-ac', '1', '-ar', '16000', str(audio)])
    output_dir = RESULTS/'diaper-overlap'/'inference'
    command = [PYTHON, str(infer_script), '-c', str(infer_config),
        '--wav-dir', str(audio_dir), '--wav-name', 'full-video',
        '--models-path', str(checkpoint.parent), '--epochs', '100',
        '--rttms-dir', str(output_dir), '--gpu', '0']
    prior_pythonpath = ENV.get('PYTHONPATH', '')
    ENV['PYTHONPATH'] = str(transformer_overlay) + os.pathsep + prior_pythonpath
    try:
        stream(command, 'diaper-overlap.log',
               'DiaPer inference failed; baseline and existing overlap results remain valid.')
    finally:
        ENV['PYTHONPATH'] = prior_pythonpath
    rttms = list(output_dir.rglob('full-video.rttm'))
    if len(rttms) != 1:
        raise RuntimeError(f'Expected one DiaPer RTTM, found {{len(rttms)}}')
    report_path = RESULTS/'diaper-overlap'/'comparison.json'
    checked([PYTHON, str(WORK/'evaluate_diaper_overlap.py'),
             '--baseline', str(RESULTS/'full_video_evidence.json'),
             '--rttm', str(rttms[0]), '--output', str(report_path)], cwd=WORK)
    return json.loads(report_path.read_text())

def export_reference_promotion_review():
    output_dir = RESULTS/'reference-promotion-review'
    manifest_path = output_dir/'manifest.json'
    if manifest_path.is_file():
        print('Reusing completed reference-promotion review:', manifest_path)
        return json.loads(manifest_path.read_text())
    if output_dir.exists():
        print('Removing incomplete reference-promotion review:', output_dir)
        shutil.rmtree(output_dir)
    command = [PYTHON, str(WORK/'reference_promotion.py'), 'export',
        '--video', str(VIDEO), '--evidence', str(RESULTS/'full_video_evidence.json'),
        '--output-dir', str(output_dir), '--source-url', VIDEO_URL,
        '--reference-metadata', str(REFERENCE/'reference.json')]
    checked(command, cwd=WORK)
    return json.loads(manifest_path.read_text())


## Run the opening check, whole video, and additive reviews

The opening check confirms that face analysis is actually using CUDA. The established stages run first. Automatic captions, when available, identify possible transcript gaps and produce short review clips after nearby wording duplicates are suppressed. Captions never identify speakers or insert text. When the matching v5 Sortformer result dataset is attached, speaker-conditioned extraction processes the union of existing baseline overlap intervals and strong Sortformer additions. MossFormer2 then separates those same review candidates, rejects weak target matches, and exports review-only evidence; it never inserts text or changes speaker identity. DiaPer remains a read-only comparison.


In [ ]:
opening_clip = extract_clip(0, 30, 'opening_30s.mp4')
opening = run_test(opening_clip, 'opening', BATCH_SIZE)
providers = opening.get('runtime', {}).get('face_providers', {})
if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
    raise RuntimeError('Face models are still on CPU. Review opening.log before starting the full video.')
print((RESULTS/'opening_transcript.txt').read_text())

if RUN_FULL_VIDEO:
    full_video = run_test(VIDEO, 'full_video', BATCH_SIZE)
    decoded_visual_segments = sum(
        1 for segment in full_video['segments']
        for evidence in segment.get('evidence', [])
        if evidence.get('source') == 'visual_context'
        and evidence.get('details', {}).get('frames_read', 0) > 0)
    if decoded_visual_segments == 0:
        raise RuntimeError('No full-video frames were decoded; supplemental reviews were not started.')
    print('Full-video segments with decoded visual frames:', decoded_visual_segments)
    confident_dir = RESULTS/'confidence-filtered-transcript'
    checked([PYTHON, str(WORK/'export_confident_transcript.py'),
             str(RESULTS/'full_video_evidence.json'),
             '--output-dir', str(confident_dir),
             '--target-minimum', '0.35', '--other-minimum', '0.65'], cwd=WORK)
    print('Confidence-filtered transcript:', confident_dir/'confident_transcript.txt')
    if RUN_CAPTION_GAP_REVIEW:
        caption_gap_review = run_caption_gap_review()
        print('Caption gap review:', json.dumps(caption_gap_review, indent=2))
    if RUN_TARGETED_REVIEW:
        targeted_review = run_targeted_review()
        print('Targeted review:', json.dumps(targeted_review, indent=2))
    if RUN_OVERLAP_EXTRACTION:
        overlap_review = run_overlap_extraction()
        print('Overlap extraction:', json.dumps(overlap_review['summary'], indent=2))
    if RUN_MOSSFORMER2_REVIEW:
        mossformer2_review = run_mossformer2_review()
        print('MossFormer2 review evidence:',
              json.dumps(mossformer2_review['summary'], indent=2))
    if RUN_DIAPER_OVERLAP:
        diaper_review = run_diaper_overlap()
        print('DiaPer overlap comparison:', json.dumps(diaper_review['summary'], indent=2))
    promotion_review = export_reference_promotion_review()
    print('Reference promotion candidates:', len(promotion_review['candidates']))
else:
    print('Full video disabled. Review the opening result, then set RUN_FULL_VIDEO=True.')


## Download results

`diarization-results.zip` contains the preserved baseline, caption-gap review clips when captions are available, existing supplemental reviews, MossFormer2 review-only evidence, DiaPer RTTM and comparison report, logs, and package versions. `stage-checkpoints.zip` restarts the complete workflow. `overlap-extraction-checkpoints.zip` and `mossformer2-checkpoints.zip` are refreshed every five windows and can be attached directly if Kaggle stops during either long stage.


In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
with (RESULTS/'runtime-packages.txt').open('w') as packages:
    checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
result_zip = Path(shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS))
checkpoint_zip = BASE/'stage-checkpoints.zip'
mossformer2_zip = BASE/'mossformer2-checkpoints.zip'
overlap_zip = BASE/'overlap-extraction-checkpoints.zip'
prior_cwd = Path.cwd()
try:
    os.chdir(BASE)
    display(FileLink(result_zip.name))
    if checkpoint_zip.is_file():
        display(FileLink(checkpoint_zip.name))
    if mossformer2_zip.is_file():
        display(FileLink(mossformer2_zip.name))
    if overlap_zip.is_file():
        display(FileLink(overlap_zip.name))
finally:
    os.chdir(prior_cwd)
if ON_KAGGLE:
    # Kaggle publishes everything under /kaggle/working. Keep only the four
    # downloadable archives instead of uploading the virtual environment,
    # model caches, source video, and unpacked duplicate results.
    cleanup_paths = [
        RESULTS, CACHE.parent, WORK, VENV, BASE/'bootstrap-tools',
        BASE/'huggingface-cache', BASE/'speechbrain-cache',
        BASE/'matplotlib-cache', BASE/'numba-cache', BASE/'insightface',
    ]
    for cleanup_path in cleanup_paths:
        if cleanup_path.is_dir():
            shutil.rmtree(cleanup_path, ignore_errors=True)
        elif cleanup_path.exists():
            cleanup_path.unlink()
print('Saved in:', BASE)
print('If Kaggle blocks a link, download the same ZIP from the Output panel.')
